In [0]:
# Step 1 -> Read customer table from bronze and store in df

# Step 2 -> Display the dataframe and try to identify what cleaning can be done

# Step 3 -> Apply the required transformation steps which you identified and the again store in DF

# Step 4 -> In final dataframe we will apply scd - type 1 and store in silver schema (before that create silver schema using sql)

# Step 5 -> Write a function in same notebook which cleans and transforms customers data from bronze and returns the dataframe

# Step 6 -> Write a function which can do SCD 1 if we provide parameters like dataframe, buisness key and target_table [After creating this function [scd1] we will take it to common_utils and take it to other notebooks and then reuse it in other notebooks]

# Step 7 - Fully modularise the code [Hardcoded values to json config, proper logging setup and all the reusable function should have used]

In [0]:
import pyspark.sql.functions as F
df = spark.read.table("retaildataplatform.bronze.sqlserver_customers")

In [0]:
df.groupBy("customer_id").count().alias("count").filter("count>1").display()

In [0]:
def transform_customer(df):
    df = df.withColumn("customer_name", F.lower(F.col("customer_name")))\
    .withColumn("name_parts", F.split(F.col("customer_name"),","))\
    .withColumn("first_name", F.expr("get(name_parts,1)"))\
    .withColumn("last_name", F.expr("get(name_parts,0)"))\
    .withColumn("city",F.upper(F.col("city")))\
    .withColumn("postcode", F.regexp_replace(F.col("postcode"),"\.0$", ""))\
    .withColumn("valid_from", F.from_unixtime(F.expr("try_cast(valid_from as bigint)")))\
    .withColumn ("valid_to", F.from_unixtime (F.expr("try_cast(valid_to as bigint)")))\
    .withColumn("country",F.lit("USA"))

    df = df.drop("name_parts","customer_name","file_path")

    df = df.dropDuplicates(["customer_id"])

    return df.select(
    "customer_id",
    "first_name",
    "last_name",
    "city",
    "district",
    "country",
    "lat",
    "lon",
    "postcode",
    "number",
    "loyalty_segment",
    "last_updated_timestamp",
    "valid_from",
    "valid_to",
    "state",
    "street",
    "region",
    "tax_code",
    "tax_id",
    "units_purchased",
    "unit")

In [0]:
cleaned_customer = transform_customer(df)
cleaned_customer.display()

In [0]:
if spark.catalog.tableExists("retaildataplatform.silver.customers"):
    print("Table Exists - Proceeding with SCD - 1")
else:
    print("Table does not exists - Creating the table")
    spark.sql("create schema if not exists retaildataplatform.silver")
    df.write.mode("overwrite").saveAsTable("retaildataplatform.silver.customers")